In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType
from pyspark.sql.functions import col, to_timestamp, desc, sum as _sum

In [2]:
# 1. Initialize SparkSession with Hive support if required by your HDFS cluster
spark = SparkSession.builder.appName("Total Net Amount per Customer Using Spark Structured Streaming") \
                .config("spark.sql.streaming.forceDeleteTempCheckpointLocation", "true") \
                .getOrCreate()


26/10/21 02:13:13 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/21 02:13:15 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/10/21 02:13:15 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/10/21 02:13:15 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
26/10/21 02:13:18 WARN Client: Neither spark.yarn.jars nor spark.yarn.archive is set, falling back to uploading libraries under SPARK_HOME.


In [3]:
# 2. Define Explicit Schemas (Required for File-Based Streaming Sources)
order_schema = StructType([
    StructField("transaction_id", StringType(), True),
    StructField("order_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("store_id", StringType(), True),
    StructField("amount", DoubleType(), True),
    StructField("amount_usd", DoubleType(), True),
    StructField("discount", DoubleType(), True),
    StructField("net_amount", DoubleType(), True),
    StructField("status", StringType(), True),
    StructField("payment_method", StringType(), True),
    StructField("currency", StringType(), True),
    StructField("order_ts", StringType(), True),
    StructField("ingestion_ts", StringType(), True)
])

customer_schema = StructType([
    StructField("customer_id", StringType(), True),
    StructField("first_name", StringType(), True),
    StructField("last_name", StringType(), True),
    StructField("email", StringType(), True),
    StructField("gender", StringType(), True),
    StructField("age", StringType(), True),
    StructField("country", StringType(), True),
    StructField("customer_segment", StringType(), True),
    StructField("signup_date", StringType(), True),
    StructField("total_orders", StringType(), True),
    StructField("total_spend_usd", StringType(), True),
    StructField("total_discount_usd", StringType(), True),
    StructField("last_order_id", StringType(), True),
    StructField("last_transaction_status", StringType(), True)
])

In [5]:
# 3. Read Customer Master Data as a Static DataFrame of customer_ord.csv
customer_static_df = spark.read.format("csv").option("header", "true").schema(customer_schema) \
    .load("/user/sclsepb456/customer_ord.csv")
customer_static_df.show(truncate=False)
print(f"loaded the customer static df: {customer_static_df}")
# 4. Read Order Transactions as a Streaming DataFrame from HDFS
order_streaming_df = spark.readStream.format("csv").option("header", "true").schema(order_schema) \
    .load("/user/sclsepb456/Raw/orders")

+-----------+----------+---------+----------------------+------+---+-------+----------------+-----------+------------+---------------+------------------+-------------+-----------------------+
|customer_id|first_name|last_name|email                 |gender|age|country|customer_segment|signup_date|total_orders|total_spend_usd|total_discount_usd|last_order_id|last_transaction_status|
+-----------+----------+---------+----------------------+------+---+-------+----------------+-----------+------------+---------------+------------------+-------------+-----------------------+
|C001       |Rahul     |Sharma   |rahul.sharma@email.com|Male  |32 |India  |VIP             |15-01-2022 |5           |4200.5         |300               |O1045        |SUCCESS                |
|C002       |Ananya    |Verma    |ananya.verma@email.com|Female|28 |India  |RETURNING       |10-03-2023 |4           |2150           |180               |O1046        |SUCCESS                |
|C003       |John      |Doe      |john.d

In [6]:
# 5. Filter for Successful Transactions & Apply 15-Minute Watermark
filtered_orders = order_streaming_df \
    .withColumn("order_ts_parsed", to_timestamp(col("order_ts"), "dd-MM-yyyy HH:mm")) \
    .filter(col("status") == "SUCCESS") \
    .withWatermark("order_ts_parsed", "15 minutes")


# 6. Calculate Running Total of net_amount per Customer
# Since we are outputting first_name only per micro-batch via foreachBatch, 
# we aggregate by customer_id to perform the rolling sum.
aggregated_orders = filtered_orders.groupBy("customer_id").agg(_sum("net_amount").alias("total_net_amount"))
print(aggregated_orders)

DataFrame[customer_id: string, total_net_amount: double]


In [7]:
# 7. Define the foreachBatch function to join with static data and show results
def process_micro_batch(micro_batch_df, batch_id):
    print(f"----- Processing Micro-batch id: {batch_id} -----")
    
    
    # Join streaming aggregatio with static customer dataset
    joined_df = micro_batch_df.join(
        customer_static_df,
        on="customer_id",
        how="inner"
    )
        
    # select and output only the required 'first_name' column
    result_df = joined_df.select("first_name","total_net_amount").orderBy(desc("total_net_amount"))
    result_df.show(truncate=False)


In [8]:
# 8. Start the Streaming Query using Complete output mode and ForeachBatch Sink
query = aggregated_orders.writeStream.outputMode("complete") \
    .foreachBatch(process_micro_batch) \
    .option("checkpointLocation","/user/sclsepb456/Staging/orders_debug-1/") \
    .start()



----- Processing Micro-batch id: 0 -----


+----------+----------------+
|first_name|total_net_amount|
+----------+----------------+
|Rajiv     |2750.0          |
|Henry     |2580.0          |
|Benjamin  |2400.0          |
|Oliver    |2300.0          |
|Manish    |2220.0          |
|Grace     |2120.0          |
|Michael   |2000.0          |
|Amelia    |1930.0          |
|Matthew   |1840.0          |
|Isabella  |1750.0          |
|Chloe     |1660.0          |
|Amit      |1650.0          |
|Ethan     |1620.0          |
|Samuel    |1570.0          |
|Lucas     |1480.0          |
|Jack      |1430.0          |
|Kunal     |1300.0          |
|Arjun     |1280.0          |
|Rahul     |850.0           |
|Nina      |720.0           |
+----------+----------------+
only showing top 20 rows



In [ ]:
#9. Await Termination to keep the streaming context alive
query.awaitTermination()